# Metadata

Domain: Mathematics

Subdomain: Linear Algebra

References:

- Strang, G. (2016). *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge Press. (Row reduction, the four fundamental subspaces.) https://math.mit.edu/~gs/linearalgebra/ila5/
- Lay, D. C., Lay, S. R., & McDonald, J. J. (2021). *Linear Algebra and Its Applications*, 6th ed. Pearson. ISBN 978-0-13-585115-9. (RREF, free variables, general solutions of Ax = b.) https://openlibrary.org/isbn/9780135851159
- Trefethen, L. N., & Bau, D. (1997). *Numerical Linear Algebra*. SIAM. ISBN 978-0-89871-361-9. (Exact vs. floating-point elimination, conditioning.) https://openlibrary.org/isbn/9780898713619
- Golub, G. H., & Van Loan, C. F. (2013). *Matrix Computations*, 4th ed. Johns Hopkins University Press. ISBN 978-1-4214-0794-4. (Gaussian elimination, rank, null space computation.) https://openlibrary.org/isbn/9781421407944

# Title

Exact Rational Linear Algebra: Row Reduction, Null-Space Bases, and the General Solution of Ax = b

# Subproblem 1

## Prompt

Implement `rref(matrix)` — the **reduced row echelon form** of a matrix, computed by exact Gauss-Jordan elimination over the rationals, together with the list of its **pivot columns**.

A matrix is represented as a list of rows, each row a list of entries; every row must have the same length. Entries are exact rational numbers, represented as Python `int` or `fractions.Fraction`.

**Reduced row echelon form (RREF).** A matrix $R$ is in RREF when:

1. Every all-zero row is below every nonzero row.
2. In each nonzero row, the leftmost nonzero entry (the **pivot**) equals exactly $1$.
3. Each pivot lies strictly to the right of the pivot in the row above it.
4. Each pivot column is zero in every row other than its own pivot row.

**Scientific trap.** Condition 4 above (each pivot column zero *everywhere else*, not just below) is the entire difference between row echelon form (REF) and *reduced* row echelon form. It is easy to write elimination that only clears each column *below* its pivot — a completely valid REF algorithm — and stop there. That output looks like a correct triangular matrix and even reports the right rank, but it is not what this contract asks for, and it will disagree with `rref`'s output on any matrix that has more than one pivot row, in exactly the rows above each later pivot.

**Why exact arithmetic.** Floating-point Gaussian elimination can leave a pivot column with a residual like `1e-16` instead of exactly `0`, so equality tests on the output become unreliable and downstream code has to guess a tolerance. Representing every entry as a `Fraction` makes every intermediate and final value exact, so `R[i][j] == 0` means exactly zero, not "small."

**Uniqueness.** The RREF of a matrix is unique: no matter which sequence of valid pivot choices or elementary row operations produces it, there is exactly one matrix satisfying the four conditions above for a given input. In particular, row-permuting the input before calling `rref` must not change the returned `rref_matrix` or `pivot_cols` — the row space is unchanged, and RREF depends only on the row space.

**Rank.** `len(pivot_cols)` is the rank of `matrix`.

```python
from fractions import Fraction


def rref(matrix):
    """
    Parameters
    ----------
    matrix : list of list
        Non-empty m x n matrix, m >= 1, n >= 1. Every row must have the
        same length n. Each entry must be a Python int or a
        fractions.Fraction. A bool is not accepted (booleans are not
        numbers in this contract, even though bool is a subclass of
        int in Python). A float is not accepted either: floats cannot
        exactly represent every rational number elimination can
        produce, so allowing them would silently reintroduce the
        inexactness this function exists to avoid.

    Returns
    -------
    rref_matrix : list of list of Fraction
        The reduced row echelon form of matrix, as defined above.
        Every entry is an exact fractions.Fraction (int entries are
        promoted). Same shape (m x n) as the input.
    pivot_cols : list of int
        The column indices that contain a pivot, in strictly
        increasing order. len(pivot_cols) == rank(matrix).

    Raises
    ------
    ValueError
        If matrix is empty, if any row is empty, or if rows have
        differing lengths.
    TypeError
        If any entry is not a Python int or a Fraction. This includes
        bool and float entries.

    Precedence when a matrix is invalid in more than one way at once:
    shape is checked before entry type. A ragged matrix that also
    contains a float entry raises ValueError, not TypeError, because
    the shape check runs to completion (or fails) before any entry is
    ever inspected for its type.
    """
    ...
    return rref_matrix, pivot_cols
```

## Background

**Elementary row operations.** Gauss-Jordan elimination reaches RREF using three moves, each of which preserves the row space (and hence the solution set of any linear system the matrix represents): (1) swap two rows, (2) scale a row by a nonzero constant, (3) add a multiple of one row to another. The algorithm sweeps left to right through the columns; for each column it looks for a nonzero entry at or below the current pivot row, swaps that row into place, scales it so the pivot is exactly $1$, and then uses row operation (3) to zero out that column in *every* other row — both below (as in plain row echelon form) and above (which is what makes it *reduced*). A column with no nonzero entry available becomes a free column and is skipped; the pivot row index does not advance.

**Worked example.** Consider

$$
A = \begin{pmatrix} 2 & 4 \\ 1 & 2 \end{pmatrix}.
$$

Column $0$: row $0$ has a nonzero entry ($2$), so it becomes the pivot row; scaling by $1/2$ gives $(1, 2)$. Eliminating column $0$ from row $1$ (which is already $(1,2)$, i.e. exactly the scaled pivot row) leaves $(0, 0)$. Column $1$: no nonzero entry remains at or below row $1$, so column $1$ is free. The result is $R = \begin{pmatrix}1 & 2\\ 0 & 0\end{pmatrix}$ with `pivot_cols = [0]` — correctly reporting that $A$ has rank $1$: its second row was a multiple of its first.

**Why the pivot columns matter.** The pivot columns of $\operatorname{rref}(A)$ identify a maximal set of linearly independent columns of $A$ and, equivalently, partition the variables of $Ax = 0$ into **pivot (basic) variables** and **free variables**. Subproblem 2 builds the null space basis directly from this partition: every free variable contributes one basis vector, so $\dim\ker A = n - \operatorname{rank}(A)$, the rank-nullity theorem. This prompt's job is to hand Subproblem 2 exactly the two ingredients — `rref_matrix` and `pivot_cols` — that make that construction possible without redoing any elimination.

**Floating point vs. exact.** Repeatedly subtracting scaled rows in floating point accumulates rounding error: eliminating with a pivot like $1/3$ and then re-combining rows can leave an entry such as $5.55\times10^{-17}$ where the true value is $0$. Every test in this notebook checks *exact* equality (`==`) against `Fraction` values, which is only sound because the contract requires exact rational entries in and exact rational entries out — there is no floating point anywhere in this pipeline.

**Validation order is part of the contract.** When an input is invalid in more than one way at once, which exception fires is not arbitrary: shape (non-empty, rectangular) is fully checked *before* any entry is inspected for its type. So a ragged matrix that also happens to contain a `float` raises `ValueError`, never `TypeError` — the shape check fails before the type-checking loop ever runs. This mirrors a real API-design question (do you validate structure or content first?) rather than being an incidental implementation detail, which is why it is specified and tested explicitly rather than left for whichever loop happens to run first in some particular implementation.

**Common bugs.**

- Stopping at row echelon form (zeros only below each pivot) without also clearing entries *above* each pivot — that is REF, not RREF, and it breaks the free-variable construction in Subproblem 2.
- Leaving the pivot entry as something other than exactly `Fraction(1)` (e.g. not dividing the pivot row by its pivot value).
- Advancing the pivot-row counter even when a column turns out to have no available nonzero entry (this desynchronizes `pivot_cols` from the actual pivot rows).
- Using `float` division instead of `Fraction` division, which reintroduces rounding error the whole point of this contract is to avoid.
- Forgetting that `bool` is technically an `int` subclass in Python and letting `True`/`False` slip through as `1`/`0` instead of raising `TypeError`.
- Off-by-one indexing between `pivot_cols[i]` and the row it actually pivots in — this corrupts the free-variable back-substitution used throughout Subproblem 2 and the Main Problem.

## Testing Template

In [ ]:
# ============ TEST CASES ============

from fractions import Fraction
from itertools import permutations


def test_case_1():
    R, pc = rref([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
    assert R == [[1, 0, 0], [0, 1, 0], [0, 0, 1]], f"got {R}"
    assert pc == [0, 1, 2], f"got {pc}"
    print('test_case_1 passed')


def test_case_2():
    # second row is a multiple of the first: rank 1, one free column.
    R, pc = rref([[2, 4], [1, 2]])
    assert R == [[1, 2], [0, 0]], f"got {R}"
    assert pc == [0], f"got {pc}"
    print('test_case_2 passed')


def test_case_3():
    # more rows than columns, full column rank: a zero row at the bottom.
    R, pc = rref([[1, 0], [0, 1], [1, 1]])
    assert R == [[1, 0], [0, 1], [0, 0]], f"got {R}"
    assert pc == [0, 1], f"got {pc}"
    print('test_case_3 passed')


def test_case_4():
    # pivots that are not integers force genuine fraction arithmetic.
    R, pc = rref([[3, 2], [1, 4]])
    assert R == [[1, 0], [0, 1]], f"got {R}"
    assert pc == [0, 1]
    print('test_case_4 passed')


def test_case_5():
    # a free (non-pivot) column in the interior of the matrix.
    R, pc = rref([[1, 2, 3], [2, 4, 9]])
    assert R == [[1, 2, 0], [0, 0, 1]], f"got {R}"
    assert pc == [0, 2], f"got {pc}"
    print('test_case_5 passed')


def test_case_6():
    R, pc = rref([[0, 0], [0, 0]])
    assert R == [[0, 0], [0, 0]]
    assert pc == []
    print('test_case_6 passed')


def test_case_7():
    R, pc = rref([[2, 4, 6]])
    assert R == [[1, 2, 3]]
    assert pc == [0]
    print('test_case_7 passed')


def test_case_8():
    # column 0 is zero in row 0, forcing a row swap to find the pivot.
    R, pc = rref([[0, 1], [1, 0]])
    assert R == [[1, 0], [0, 1]]
    assert pc == [0, 1]
    print('test_case_8 passed')


def test_case_9():
    # Fraction entries in the input, not just int.
    R, pc = rref([[Fraction(1, 2), 1], [1, 2]])
    assert R == [[1, 2], [0, 0]], f"got {R}"
    assert pc == [0]
    print('test_case_9 passed')


def test_case_10():
    for M, expected_R, expected_pc in [
        ([[1, 1, 1, 1]], [[1, 1, 1, 1]], [0]),
        ([[1, 1, 2], [2, 2, 4]], [[1, 1, 2], [0, 0, 0]], [0]),
    ]:
        R, pc = rref(M)
        assert R == expected_R, f"{M}: got {R}, expected {expected_R}"
        assert pc == expected_pc, f"{M}: got {pc}, expected {expected_pc}"
    print('test_case_10 passed')


def test_case_11():
    # structural sanity, checked generically across several matrices:
    # pivot columns strictly increasing, every pivot entry exactly 1,
    # and each pivot column zero everywhere except its own pivot row.
    for M in [
        [[1, 2], [3, 4]],
        [[2, 4], [1, 2]],
        [[1, 2, 3], [4, 5, 6], [7, 8, 10]],
        [[0, 2], [3, 0]],
        [[5]],
    ]:
        R, pc = rref(M)
        assert pc == sorted(pc) and len(set(pc)) == len(pc), f"{M}: pc={pc}"
        for i, col in enumerate(pc):
            assert R[i][col] == 1, f"{M}: pivot at row {i} col {col} is {R[i][col]}, expected 1"
            for row_idx, row in enumerate(R):
                if row_idx != i:
                    assert row[col] == 0, (
                        f"{M}: column {col} has a nonzero entry {row[col]} "
                        f"in non-pivot row {row_idx}"
                    )
    print('test_case_11 passed')


def test_case_12():
    # RREF is unique: permuting the input rows must not change the output.
    M = [[1, 2, 3], [2, 4, 7], [1, 1, 2]]
    base_R, base_pc = rref(M)
    for perm in permutations(range(len(M))):
        R, pc = rref([M[i] for i in perm])
        assert R == base_R, f"perm {perm}: R changed"
        assert pc == base_pc, f"perm {perm}: pivot_cols changed"
    print('test_case_12 passed')


def test_case_13():
    for bad in [[], [[]], [[1, 2], [1]]]:
        try:
            rref(bad)
            assert False, f"expected ValueError for {bad}"
        except ValueError:
            pass
    print('test_case_13 passed')


def test_case_14():
    for bad_matrix in [[[1.0, 2], [3, 4]], [[True, 2], [3, 4]], [["1", 2], [3, 4]]]:
        try:
            rref(bad_matrix)
            assert False, f"expected TypeError for {bad_matrix}"
        except TypeError:
            pass
    print('test_case_14 passed')


def test_case_15():
    # precedence: shape is checked before entry type. A ragged matrix that
    # ALSO has a float entry must still raise ValueError, not TypeError --
    # the shape check never even reaches the offending entry.
    try:
        rref([[1.0, 2], [3]])
        assert False, "expected ValueError"
    except ValueError:
        pass
    except TypeError:
        assert False, "TypeError fired instead of ValueError -- wrong precedence"
    print('test_case_15 passed')


def test_case_16():
    # Scientific trap: an implementation that only clears BELOW each pivot
    # (row echelon form) instead of both above and below (reduced row
    # echelon form) will get the pivot positions and the rank right, but
    # will leave nonzero entries above the second pivot -- exactly what
    # this input is built to expose.
    M = [[2, 4, -2, 2],
         [1, 1, 3, 0],
         [3, 5, 1, 2]]
    R, pc = rref(M)
    expected_R = [[1, 0, 7, -1], [0, 1, -4, 1], [0, 0, 0, 0]]
    assert R == expected_R, f"got {R}, expected {expected_R} (check clearing ABOVE pivots too)"
    assert pc == [0, 1], f"got {pc}"
    print('test_case_16 passed')


def test_case_17():
    # a larger matrix: 3 pivots, 2 free columns, and fraction (non-integer)
    # pivot arithmetic throughout -- large enough that a single lucky
    # coincidence can't make a wrong implementation pass by accident.
    M = [[1, 2, 3, 0, 4],
         [2, 4, 8, 1, 10],
         [3, 6, 7, -1, 9]]
    R, pc = rref(M)
    expected_R = [
        [1, 2, 0, Fraction(-3, 2), 0],
        [0, 0, 1, Fraction(1, 2), 0],
        [0, 0, 0, 0, 1],
    ]
    assert R == expected_R, f"got {R}"
    assert pc == [0, 2, 4], f"got {pc}"
    print('test_case_17 passed')

## Solution

In [ ]:
from fractions import Fraction


def rref(matrix):
    """
    Reduced row echelon form of matrix over the exact rationals, together
    with the list of pivot columns.
    """
    if not isinstance(matrix, (list, tuple)) or len(matrix) == 0:
        raise ValueError("matrix must be a non-empty list of rows")

    n_cols = None
    for row in matrix:
        if not isinstance(row, (list, tuple)) or len(row) == 0:
            raise ValueError("each row must be a non-empty list")
        if n_cols is None:
            n_cols = len(row)
        elif len(row) != n_cols:
            raise ValueError("all rows must have the same length")

    m, n = len(matrix), n_cols

    R = []
    for row in matrix:
        new_row = []
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, Fraction)):
                raise TypeError(
                    "matrix entries must be int or Fraction (no float/bool)"
                )
            new_row.append(Fraction(x))
        R.append(new_row)

    pivot_cols = []
    r = 0
    for c in range(n):
        # find the topmost row at or below r with a nonzero entry in column c
        pivot = None
        for i in range(r, m):
            if R[i][c] != 0:
                pivot = i
                break
        if pivot is None:
            continue  # free column: no pivot here, r does not advance

        R[r], R[pivot] = R[pivot], R[r]
        pv = R[r][c]
        R[r] = [x / pv for x in R[r]]  # scale pivot row so pivot == 1

        for i in range(m):
            if i != r and R[i][c] != 0:
                factor = R[i][c]
                R[i] = [a - factor * b for a, b in zip(R[i], R[r])]

        pivot_cols.append(c)
        r += 1
        if r == m:
            break

    return R, pivot_cols

# Subproblem 2

## Prompt

Implement `null_space_basis(rref_matrix, pivot_cols)` — a basis for the **null space** (kernel) of a matrix, given its reduced row echelon form and pivot columns in the canonical form produced by `rref` (see the contract restated below; you do not need to implement or call `rref` yourself — every test below supplies a complete, ready-made `rref_matrix`/`pivot_cols` pair).

**The free-variable construction.** Write $R$ for `rref_matrix`, an $m \times n$ matrix, and $(p_0, p_1, \dots, p_{r-1})$ for `pivot_cols`, listed in row order: $p_i$ is the pivot column of **row $i$** of $R$ (not the other way around — $p_i$ is a *column* index and $i$ is a *row* index, and outside of the trivial case $p_i = i$ they are different numbers; see the trap below). Every column index not among the $p_i$ is a **free column**. For each free column $f$, in increasing order of $f$, build the vector $v \in \mathbb{Q}^n$ by:

- setting $v_f = 1$ and $v_{f'} = 0$ for every *other* free column $f'$,
- setting $v_{p_i} = -R_{i,f}$ for **every** $i$ from $0$ to $r-1$ — note that the value comes from **row $i$**, and is stored at **column $p_i$**; these are generally two different indices.

The resulting $v$ satisfies $Rv = 0$ because row $i$ of $R$ reads $x_{p_i} + \sum_{f \text{ free}} R_{i,f}\, x_f = 0$, and this construction is exactly the statement "set free variable $f$ to $1$, every other free variable to $0$, and back-solve the pivot variables." One such $v$ per free column gives `n - len(pivot_cols)` vectors, and by rank-nullity these form a basis of $\ker R$ — which equals $\ker A$ for any matrix $A$ whose RREF is $R$, since elementary row operations never change the solution set of a homogeneous system.

**This function must validate, not assume, that `rref_matrix` genuinely matches `pivot_cols`.** The construction above is only correct if four structural facts hold, for every $i$ from $0$ to $r-1$ ($r$ = `len(pivot_cols)`):

1. $R_{i,\,p_i} = 1$ exactly (row $i$'s pivot entry really is the leading $1$ the construction assumes it is).
2. $R_{i,\,j} = 0$ for every column $j < p_i$ (the declared pivot $p_i$ really is the **leftmost** nonzero entry of row $i$ — there is nothing to its left — which is the defining geometry of reduced row echelon form; without it $p_i$ would not be row $i$'s actual pivot).
3. $R_{j,\,p_i} = 0$ for every row $j \neq i$ (column $p_i$ is really a pivot column — clean everywhere except its own pivot row).
4. Every row $j \geq r$ (beyond the declared rank) is entirely zero (there is no hidden, undeclared pivot lurking below the ones `pivot_cols` reports).

A caller who passes a `pivot_cols` that does not actually match `rref_matrix` this way is not describing a valid reduced row echelon form, and the free-variable formula above is meaningless applied to it — it would silently produce some vector, but that vector would not satisfy $Rv=0$, and the whole downstream guarantee (this is a *basis of the null space*) would be false. This function must detect that and raise `ValueError`, not return a wrong answer.

**Scientific trap 1 (sign).** It is tempting to set $v_{p_i} = R_{i,f}$ (no minus sign) by analogy with "just read the row off directly." That produces a vector satisfying $Rv = 2R_{\bullet,f}$ in general — nonzero — not $Rv=0$. The minus sign comes directly from solving $x_{p_i} + R_{i,f}\,x_f = 0$ for $x_{p_i}$, and any test that actually substitutes the returned vector back into $R$ (rather than only checking its shape or its entry count) will catch a dropped sign immediately.

**Scientific trap 2 (row index vs. column index).** `pivot_cols[i]` is a *column* index; `i` itself is the *row* index of that pivot. When there is even one free column before a pivot, `pivot_cols[i] != i` — e.g. `pivot_cols = [1, 3]` means row $0$ pivots on column $1$ and row $1$ pivots on column $3$; column $0$ is a leading free column, not a shifted-over pivot. An implementation that conflates the two — for instance indexing `rref_matrix[pivot_cols[i]]` where it should index `rref_matrix[i]`, or writing the back-solved value into row-position `i` of the vector instead of column-position `pivot_cols[i]` — will pass every test where `pivot_cols == list(range(len(pivot_cols)))` (pivots crowded at the very first columns) and silently fail everywhere else.

**Exception policy — the same type/value split as `rref`, now with one more tier.** `TypeError` for the wrong *type* of an entry or index. `ValueError` for the right type but an invalid *value*, and this now covers two genuinely different kinds of invalidity: `pivot_cols` being malformed as a list of indices (duplicate, unsorted, out of range, too many entries) **and** `rref_matrix` failing to structurally match a valid `pivot_cols` (conditions 1-4 above). Both are `ValueError`; see Raises below for the precedence between all of these when more than one is violated at once.

```python
def null_space_basis(rref_matrix, pivot_cols):
    """
    Parameters
    ----------
    rref_matrix : list of list
        An m x n matrix. Entries must be int or Fraction. Must
        genuinely be consistent with pivot_cols as a reduced row
        echelon form (see conditions 1-4 above) -- this is checked,
        not assumed.
    pivot_cols : list of int
        The pivot columns, in row order: pivot_cols[i] is the pivot
        column of row i. Must be distinct, sorted in strictly
        increasing order, each a valid column index (0 <= col < n),
        with len(pivot_cols) <= m. Entries must be Python int (bool
        excluded).

    Returns
    -------
    basis : list of list of Fraction
        One vector per free column, each of length n, in increasing
        order of the free column it corresponds to. Every entry is an
        exact fractions.Fraction. Empty list when there are no free
        columns (rref_matrix has full column rank).

    Raises
    ------
    TypeError
        If any entry of rref_matrix is not int or Fraction (this
        includes bool and float), or if pivot_cols is not a list/tuple
        of Python int (this includes bool).
    ValueError
        If rref_matrix is empty, if any of its rows is empty, or if
        its rows have differing lengths; or if pivot_cols contains a
        duplicate, an out-of-range column index, is not sorted in
        increasing order, or has more entries than rref_matrix has
        rows; or if rref_matrix does not structurally match pivot_cols
        (some rref_matrix[i][pivot_cols[i]] != 1, some other row is
        nonzero in a pivot column, or some row at or beyond
        len(pivot_cols) is not entirely zero).

    Precedence, when more than one problem is present at once, from
    highest priority to lowest:
      1. rref_matrix's own shape (empty / ragged) -- ValueError.
      2. rref_matrix's entry types -- TypeError.
      3. pivot_cols's own type and well-formedness as a plain list of
         indices (type, duplicates, sort order, range, count) --
         TypeError or ValueError.
      4. Whether rref_matrix structurally matches pivot_cols as an
         RREF (conditions 1-4 above) -- ValueError.
    Each tier is only checked once every higher tier has passed --
    for example, a duplicate in pivot_cols (tier 3) is reported even
    if rref_matrix would also fail the structural match in tier 4,
    because tier 3 is checked first and pivot_cols cannot even be used
    to index into rref_matrix meaningfully until it passes tier 3.
    """
    ...
    return basis
```

## Background

**The null space.** For an $m \times n$ matrix $A$, $\ker A = \{x \in \mathbb{Q}^n : Ax = 0\}$ is a subspace of $\mathbb{Q}^n$. Elementary row operations change $A$ but never change which vectors solve $Ax = 0$ (a row operation replaces one linear combination of the equations with another, equivalent, set of equations), so $\ker A = \ker \operatorname{rref}(A)$ exactly. This is why `null_space_basis` only ever needs `rref_matrix` and `pivot_cols` — it never needs to see the original $A$.

**Why the construction gives a basis, not just a spanning set.** Look at the free-column coordinates of the returned vectors: the $k$-th basis vector has a $1$ in its own free column and a $0$ in every *other* free column, by construction. Restricted to just the free-column coordinates, the vectors therefore form the standard basis of $\mathbb{Q}^{n-r}$ ($r = \operatorname{rank}$) — which is obviously linearly independent — and the pivot-column coordinates are then *determined* by the free ones, so the full set inherits that independence. Spanning follows because every solution of $Rx=0$ is uniquely determined by its free-variable values (the pivot variables are forced), so every solution is some linear combination of these particular basis vectors, one coefficient per free variable.

**Why validating structure is not optional.** `rref_matrix` and `pivot_cols` are two separate arguments, supplied by whoever calls this function. Nothing in Python stops a caller from passing a `pivot_cols` that simply does not describe `rref_matrix` correctly — that is a bug in the *caller*, not a type error, so no `isinstance` check can catch it. Consider $R = \begin{pmatrix}2 & 5\\0&0\end{pmatrix}$ with a claimed `pivot_cols = [0]`. Blindly applying the free-variable formula to free column $1$ gives $v_0 = -R_{0,1} = -5$ and $v_1 = 1$, i.e. $v = (-5, 1)$ — but $Rv = (2\cdot(-5) + 5\cdot 1,\ 0) = (-5, 0) \neq (0,0)$. The formula assumes $R_{0,0} = 1$; here it is $2$, and the returned "basis vector" is not in the null space at all. This is the direct, mechanical consequence of applying the formula outside the conditions under which it was derived, not a corner case invented for the sake of having a test.

Note that this bug is invisible on any input with a *trivial* null space (no free columns): a matrix like $\begin{pmatrix}2&0\\0&1\end{pmatrix}$ with the same malformed `pivot_cols = [0, 1]` returns `[]` regardless of whether the pivot-value check is implemented, since there is nothing to back-solve. This is exactly why every trap test below pairs a structural defect with a *nontrivial* null space — otherwise the defect has nowhere to show up.

**Scientific trap, worked out numerically (row index vs. column index).** Take the genuine RREF $R = \begin{pmatrix} 0 & 1 & 2 & 0 & -1 \\ 0 & 0 & 0 & 1 & 3 \end{pmatrix}$ with `pivot_cols = [1, 3]`: row $0$ pivots on column $1$, row $1$ pivots on column $3$; columns $0, 2, 4$ are free, and column $0$ is a *leading* free column (so $p_0 = 1 \neq 0$ — the row and column indices genuinely differ). The correct basis vector for free column $2$ sets $v_2 = 1$ and back-solves $v_1 = -R_{0,2} = -2$ (from **row 0**, stored at **column 1**) and $v_3 = -R_{1,2} = 0$ (from **row 1**, stored at **column 3**), giving $(0,-2,1,0,0)$ — and indeed $R \cdot (0,-2,1,0,0) = (-2 + 2,\ 0) = (0,0)$. An implementation that conflates row and column index here — e.g. reading `rref_matrix[1][...]` where it means row $0$'s data, because it absent-mindedly used the pivot *column number* $1$ as a row index — would instead read row $1$'s entries, giving a vector that looks equally plausible (same shape, same nonzero pattern) but is not in $\ker R$ at all. Nothing about the vector's shape reveals the bug; only substituting it back into $R$ does.

**Rank-nullity.** $\dim\ker A + \operatorname{rank} A = n$ (the number of columns), i.e. `len(basis) == n - len(pivot_cols)`. This is the linear-algebra analogue of the Euler-characteristic check used elsewhere: it is a genuinely independent way to catch a wrong basis size without knowing the individual vectors — though, as the worked example above shows, it is *not* enough on its own to catch every bug, since a structurally wrong vector can still have the right count.

**Validation order is part of the contract, and now has four tiers.** When several things are wrong at once, which exception fires is specified, not arbitrary: (1) `rref_matrix`'s own shape, then (2) its entry types, are fully resolved before `pivot_cols` is inspected at all; then (3) `pivot_cols`'s well-formedness as a plain list of indices (duplicates, sort order, range, count) is resolved before (4) the deeper question of whether `rref_matrix` actually, structurally, matches `pivot_cols` as a genuine RREF is even asked — because that question only makes sense once `pivot_cols` is known to be a sane list of indices to index with in the first place. A duplicate in `pivot_cols` is reported even when `rref_matrix` would also fail the structural match, since tier 3 runs first.

**Common bugs.**

- Dropping the minus sign in $v_{p_i} = -R_{i,f}$ (produces vectors satisfying $Rv = 2 \cdot(\text{something})$, not $0$, in general).
- Conflating the row index $i$ with the pivot column index $p_i = $ `pivot_cols[i]` — indexing `rref_matrix` by the wrong one of the two. Invisible whenever `pivot_cols == [0, 1, 2, ...]`, which is why every test below with more than one pivot deliberately includes at least one case where `pivot_cols[i] != i`.
- Skipping the structural validation entirely (assuming `rref_matrix` and `pivot_cols` are already consistent) — this doesn't just miss an edge case, it silently returns vectors that are not in the null space at all whenever the caller's two arguments disagree, exactly as in the worked numerical example above.
- Checking only that each declared pivot entry is *nonzero*, instead of *exactly $1$* — the free-variable formula assumes the pivot coefficient already is $1$, so a pivot entry of, say, $2$, silently produces a vector off by a factor of $2$ in that coordinate rather than raising an error.
- Checking that each pivot c**The null space.** For an $m \times n$ matrix $A$, $\ker A = \{x \in \mathbb{Q}^n : Ax = 0\}$ is a subspace of $\mathbb{Q}^n$. Elementary row operations change $A$ but never change which vectors solve $Ax = 0$ (a row operation replaces one linear combination of the equations with another, equivalent, set of equations), so $\ker A = \ker \operatorname{rref}(A)$ exactly. This is why `null_space_basis` only ever needs `rref_matrix` and `pivot_cols` — it never needs to see the original $A$.

**Why the construction gives a basis, not just a spanning set.** Look at the free-column coordinates of the returned vectors: the $k$-th basis vector has a $1$ in its own free column and a $0$ in every *other* free column, by construction. Restricted to just the free-column coordinates, the vectors therefore form the standard basis of $\mathbb{Q}^{n-r}$ ($r = \operatorname{rank}$) — which is obviously linearly independent — and the pivot-column coordinates are then *determined* by the free ones, so the full set inherits that independence. Spanning follows because every solution of $Rx=0$ is uniquely determined by its free-variable values (the pivot variables are forced), so every solution is some linear combination of these particular basis vectors, one coefficient per free variable.

**Why validating structure is not optional.** `rref_matrix` and `pivot_cols` are two separate arguments, supplied by whoever calls this function. Nothing in Python stops a caller from passing a `pivot_cols` that simply does not describe `rref_matrix` correctly — that is a bug in the *caller*, not a type error, so no `isinstance` check can catch it. Consider $R = \begin{pmatrix}2 & 5\\0&0\end{pmatrix}$ with a claimed `pivot_cols = [0]`. Blindly applying the free-variable formula to free column $1$ gives $v_0 = -R_{0,1} = -5$ and $v_1 = 1$, i.e. $v = (-5, 1)$ — but $Rv = (2\cdot(-5) + 5\cdot 1,\ 0) = (-5, 0) \neq (0,0)$. The formula assumes $R_{0,0} = 1$; here it is $2$, and the returned "basis vector" is not in the null space at all. This is the direct, mechanical consequence of applying the formula outside the conditions under which it was derived, not a corner case invented for the sake of having a test.

Note that this bug is invisible on any input with a *trivial* null space (no free columns): a matrix like $\begin{pmatrix}2&0\\0&1\end{pmatrix}$ with the same malformed `pivot_cols = [0, 1]` returns `[]` regardless of whether the pivot-value check is implemented, since there is nothing to back-solve. This is exactly why every trap test below pairs a structural defect with a *nontrivial* null space — otherwise the defect has nowhere to show up.

**Scientific trap, worked out numerically (row index vs. column index).** Take the genuine RREF $R = \begin{pmatrix} 0 & 1 & 2 & 0 & -1 \\ 0 & 0 & 0 & 1 & 3 \end{pmatrix}$ with `pivot_cols = [1, 3]`: row $0$ pivots on column $1$, row $1$ pivots on column $3$; columns $0, 2, 4$ are free, and column $0$ is a *leading* free column (so $p_0 = 1 \neq 0$ — the row and column indices genuinely differ). The correct basis vector for free column $2$ sets $v_2 = 1$ and back-solves $v_1 = -R_{0,2} = -2$ (from **row 0**, stored at **column 1**) and $v_3 = -R_{1,2} = 0$ (from **row 1**, stored at **column 3**), giving $(0,-2,1,0,0)$ — and indeed $R \cdot (0,-2,1,0,0) = (-2 + 2,\ 0) = (0,0)$. An implementation that conflates row and column index here — e.g. reading `rref_matrix[1][...]` where it means row $0$'s data, because it absent-mindedly used the pivot *column number* $1$ as a row index — would instead read row $1$'s entries, giving a vector that looks equally plausible (same shape, same nonzero pattern) but is not in $\ker R$ at all. Nothing about the vector's shape reveals the bug; only substituting it back into $R$ does.

**Rank-nullity.** $\dim\ker A + \operatorname{rank} A = n$ (the number of columns), i.e. `len(basis) == n - len(pivot_cols)`. This is the linear-algebra analogue of the Euler-characteristic check used elsewhere: it is a genuinely independent way to catch a wrong basis size without knowing the individual vectors — though, as the worked example above shows, it is *not* enough on its own to catch every bug, since a structurally wrong vector can still have the right count.

**Validation order is part of the contract, and now has four tiers.** When several things are wrong at once, which exception fires is specified, not arbitrary: (1) `rref_matrix`'s own shape, then (2) its entry types, are fully resolved before `pivot_cols` is inspected at all; then (3) `pivot_cols`'s well-formedness as a plain list of indices (duplicates, sort order, range, count) is resolved before (4) the deeper question of whether `rref_matrix` actually, structurally, matches `pivot_cols` as a genuine RREF is even asked — because that question only makes sense once `pivot_cols` is known to be a sane list of indices to index with in the first place. A duplicate in `pivot_cols` is reported even when `rref_matrix` would also fail the structural match, since tier 3 runs first.

**Common bugs.**

- Dropping the minus sign in $v_{p_i} = -R_{i,f}$ (produces vectors satisfying $Rv = 2 \cdot(\text{something})$, not $0$, in general).
- Conflating the row index $i$ with the pivot column index $p_i = $ `pivot_cols[i]` — indexing `rref_matrix` by the wrong one of the two. Invisible whenever `pivot_cols == [0, 1, 2, ...]`, which is why every test below with more than one pivot deliberately includes at least one case where `pivot_cols[i] != i`.
- Skipping the structural validation entirely (assuming `rref_matrix` and `pivot_cols` are already consistent) — this doesn't just miss an edge case, it silently returns vectors that are not in the null space at all whenever the caller's two arguments disagree, exactly as in the worked numerical example above.
- Checking only that each declared pivot entry is *nonzero*, instead of *exactly $1$* — the free-variable formula assumes the pivot coefficient already is $1$, so a pivot entry of, say, $2$, silently produces a vector off by a factor of $2$ in that coordinate rather than raising an error.
- Checking that each pivot column is clean in rows *below* its own pivot row (row-echelon style) but forgetting rows *above* it — a pivot column with a stray nonzero entry above its pivot row is just as much a violation of "clean everywhere else" as one below.
- Forgetting to check that rows at or beyond `len(pivot_cols)` are all zero — a caller could under-report the rank (leave a genuine pivot row out of `pivot_cols`), and without this check that row's actual constraint on the null space is silently dropped.
- Iterating over free columns in the wrong order, so the returned basis doesn't match the "in increasing order of free column" contract (harmless mathematically, but breaks exact-equality tests and any downstream code that indexes into the result).
- Returning `[()]` or `[[]]` instead of `[]` when the null space is trivial.olumn is clean in rows *below* its own pivot row (row-echelon style) but forgetting rows *above* it — a pivot column with a stray nonzero entry above its pivot row is just as much a violation of "clean everywhere else" as one below.
- Forgetting to check that rows at or beyond `len(pivot_cols)` are all zero — a caller could under-report the rank (leave a genuine pivot row out of `pivot_cols`), and without this check that row's actual constraint on the null space is silently dropped.
- Iterating over free columns in the wrong order, so the returned basis doesn't match the "in increasing order of free column" contract (harmless mathematically, but breaks exact-equality tests and any downstream code that indexes into the result).
- Returning `[()]` or `[[]]` instead of `[]` when the null space is trivial.

## Testing Template

In [ ]:
# ============ TEST CASES ============
#
# These rref_matrix / pivot_cols pairs are complete, hand-built inputs.
# null_space_basis is never given anything produced by calling rref() --
# every test constructs its own canonical-form input directly.

from fractions import Fraction


def _matvec(R, v):
    return [sum(Fraction(a) * Fraction(x) for a, x in zip(row, v)) for row in R]


def _is_zero_vector(w):
    return all(x == 0 for x in w)


def test_case_1():
    """Identity: trivial null space, empty basis."""
    assert null_space_basis([[1, 0, 0], [0, 1, 0], [0, 0, 1]], [0, 1, 2]) == []
    print('test_case_1 passed')


def test_case_2():
    """One free column; check the exact vector."""
    basis = null_space_basis([[1, 0, -2], [0, 1, 3]], [0, 1])
    assert basis == [[2, -3, 1]], f"got {basis}"
    print('test_case_2 passed')


def test_case_3():
    """Zero matrix: every column free, basis is the standard basis."""
    basis = null_space_basis([[0, 0, 0]], [])
    assert basis == [[1, 0, 0], [0, 1, 0], [0, 0, 1]], f"got {basis}"
    print('test_case_3 passed')


def test_case_4():
    """Trailing zero row must not disturb the pivot-row correspondence."""
    basis = null_space_basis([[1, 0, 3], [0, 1, -1], [0, 0, 0]], [0, 1])
    assert basis == [[-3, 1, 1]], f"got {basis}"
    print('test_case_4 passed')


def test_case_5():
    """Two free columns, basis in increasing order of free column."""
    basis = null_space_basis([[1, 0, 2, -1], [0, 1, -3, 4]], [0, 1])
    assert basis == [[-2, 3, 1, 0], [1, -4, 0, 1]], f"got {basis}"
    print('test_case_5 passed')


def test_case_6():
    """Full column rank: trivial null space; 1x2 with one pivot has a
    single free column."""
    assert null_space_basis([[1, 0], [0, 1]], [0, 1]) == []
    basis = null_space_basis([[1, 0]], [0])
    assert basis == [[0, 1]], f"got {basis}"
    print('test_case_6 passed')


def test_case_7():
    """Fraction entries produce fraction-valued basis vectors."""
    basis = null_space_basis([[1, Fraction(1, 3)], [0, 0]], [0])
    assert basis == [[Fraction(-1, 3), 1]], f"got {basis}"
    print('test_case_7 passed')


def test_case_9():
    """Sign trap: the minus sign in v[p_i] = -R[i][f] must be present."""
    R = [[1, 0, 5], [0, 1, -2]]
    basis = null_space_basis(R, [0, 1])
    assert basis == [[-5, 2, 1]], f"got {basis} (check the minus sign)"
    print('test_case_9 passed')


def test_case_10():
    """Row-index vs column-index trap: pivot_cols[0]=1, pivot_cols[1]=3,
    so p_0 != 0 and p_1 != 1 -- an implementation that reads row 1's
    data for row 0 produces the wrong vector."""
    R = [[0, 1, 2, 0, -1], [0, 0, 0, 1, 3]]
    pc = [1, 3]
    basis = null_space_basis(R, pc)
    expected = [
        [1, 0, 0, 0, 0],
        [0, -2, 1, 0, 0],
        [0, 1, 0, -3, 1],
    ]
    assert basis == expected, f"got {basis}"
    for v in basis:
        assert _is_zero_vector(_matvec(R, v)), f"R @ {v} != 0"
    print('test_case_10 passed')


def test_case_11():
    """Larger interleaved case (3x6)."""
    R = [[0, 1, 7, 0, -2, 0], [0, 0, 0, 1, 4, 0], [0, 0, 0, 0, 0, 1]]
    pc = [1, 3, 5]
    basis = null_space_basis(R, pc)
    assert len(basis) == 6 - len(pc) == 3, f"got {len(basis)}"
    for v in basis:
        assert _is_zero_vector(_matvec(R, v)), f"R @ {v} != 0"
    print('test_case_11 passed')


def test_case_12():
    """Malformed pivot_cols: duplicate / unsorted / out-of-range / too many."""
    R = [[1, 0, -2], [0, 1, 3]]
    for bad_pc in ([0, 0], [1, 0], [0, 5], [0, 1, 2, 3]):
        try:
            null_space_basis(R, bad_pc)
            assert False, f"expected ValueError for pivot_cols={bad_pc}"
        except ValueError:
            pass
    print('test_case_12 passed')


def test_case_13():
    """bool and float entries -> TypeError; bools in pivot_cols -> TypeError."""
    for bad_pc in ([0, 1.0], [0, True], ["0", 1], 0):
        try:
            null_space_basis([[1, 0, -2], [0, 1, 3]], bad_pc)
            assert False, f"expected TypeError for pivot_cols={bad_pc!r}"
        except TypeError:
            pass
    for bad_R in ([[1.0, 0], [0, 1]], [[True, 0], [0, 1]]):
        try:
            null_space_basis(bad_R, [0, 1])
            assert False, f"expected TypeError for rref_matrix={bad_R}"
        except TypeError:
            pass
    print('test_case_13 passed')


def test_case_14():
    """Tuples are accepted for rref_matrix, its rows, and pivot_cols, and
    produce identical results to the list-of-list form."""
    expected = [[2, -3, 1]]
    for R, pc in [
        ([[1, 0, -2], [0, 1, 3]], [0, 1]),
        (([1, 0, -2], [0, 1, 3]), [0, 1]),
        (((1, 0, -2), (0, 1, 3)), [0, 1]),
        ([[1, 0, -2], [0, 1, 3]], (0, 1)),
    ]:
        basis = null_space_basis(R, pc)
        assert basis == expected, f"R={R!r}, pc={pc!r}: got {basis}"
    print('test_case_14 passed')


def test_case_15():
    """Non-iterable rref_matrix -> ValueError, not an uncaught TypeError
    from len()."""
    for bad in (5, 2.5, None):
        try:
            null_space_basis(bad, [])
            assert False, f"expected ValueError for rref_matrix={bad!r}"
        except ValueError:
            pass
        except TypeError:
            assert False, (
                f"TypeError fired for rref_matrix={bad!r}; the shape check "
                f"should run first and raise ValueError"
            )
    print('test_case_15 passed')


def test_case_16():
    """String rref_matrix -> ValueError: a str is not a valid matrix shape,
    so the shape tier fires before any entry-type check."""
    try:
        null_space_basis("abc", [])
        assert False, "expected ValueError for rref_matrix='abc'"
    except ValueError:
        pass
    except TypeError:
        assert False, (
            "TypeError fired for rref_matrix='abc'; the shape check "
            "should run first and raise ValueError"
        )
    print('test_case_16 passed')


def test_case_17():
    """Structural mismatch between rref_matrix and pivot_cols -> ValueError."""
    for bad_R, bad_pc in [
        ([[2, 3], [0, 1]], [0, 1]),                                # pivot != 1
        ([[1, 0], [3, 1]], [0, 1]),                                # stray below
        ([[1, 0, 5], [0, 1, 0], [0, 0, 1]], [0, 1, 2]),            # stray above
        ([[1, 0, 2], [0, 1, 3], [0, 0, 5]], [0, 1]),               # trailing nonzero
    ]:
        try:
            null_space_basis(bad_R, bad_pc)
            assert False, f"expected ValueError for {bad_R}, {bad_pc}"
        except ValueError:
            pass
    print('test_case_17 passed')


def test_case_18():
    """Precedence: shape (tier 1) before entry type (tier 2), entry type
    before pivot_cols (tier 3), pivot_cols before structural match (tier 4)."""
    # ragged + float entry -> ValueError (shape fires first)
    try:
        null_space_basis([[1.0, 2], [3]], [0])
        assert False
    except ValueError:
        pass
    except TypeError:
        assert False, "TypeError fired; shape check should win"

    # float entry + bool in pivot_cols -> TypeError from entry
    try:
        null_space_basis([[1.0, 0], [0, 1]], [0, True])
        assert False
    except TypeError:
        pass
    except ValueError:
        assert False, "ValueError fired; entry-type check should win"

    # duplicate pivot_cols + structural mismatch in rref_matrix -> ValueError
    try:
        null_space_basis([[2, 3], [0, 1]], [0, 0])
        assert False
    except ValueError:
        pass
    print('test_case_18 passed')


def test_case_19():
    # Condition 2 (leftmost-pivot): a declared pivot must be the leftmost nonzero entry of its
    # row. A matrix with a nonzero entry LEFT of a declared pivot is not a genuine RREF even if
    # the pivot entry is 1 and the pivot column is otherwise clean; the reference must raise
    # ValueError (it is not a valid canonical RREF / pivot_cols pair).
    for R, pc in [
        ([[1, 1]], [1]),            # column 0 (=1) is really the pivot, not column 1
        ([[2, 1]], [1]),            # nonzero 2 left of the declared pivot
        ([[0, 1, 3], [4, 0, 1]], [1, 2]),   # row 1 has 4 in column 0, left of pivot col 2
    ]:
        try:
            null_space_basis(R, pc)
            raise AssertionError(f"expected ValueError for non-leftmost pivot: R={R}, pc={pc}")
        except ValueError:
            pass
    print('test_case_19 passed')



## Solution

In [ ]:
from fractions import Fraction


def null_space_basis(rref_matrix, pivot_cols):
    """
    A basis of ker(rref_matrix) built from the free-column construction,
    given rref_matrix and pivot_cols in the canonical form rref() returns.

    Validation runs in four tiers, each only reached once the previous
    tier has passed: (1) rref_matrix's shape, (2) rref_matrix's entry
    types, (3) pivot_cols's well-formedness as a plain list of indices,
    (4) whether rref_matrix genuinely, structurally, matches pivot_cols
    as a reduced row echelon form.
    """
    # --- tier 1: rref_matrix shape ---
    if not isinstance(rref_matrix, (list, tuple)) or len(rref_matrix) == 0:
        raise ValueError("rref_matrix must be a non-empty list of rows")

    n_cols = None
    for row in rref_matrix:
        if not isinstance(row, (list, tuple)) or len(row) == 0:
            raise ValueError("each row of rref_matrix must be a non-empty list")
        if n_cols is None:
            n_cols = len(row)
        elif len(row) != n_cols:
            raise ValueError("all rows of rref_matrix must have the same length")

    m, n = len(rref_matrix), n_cols

    # --- tier 2: rref_matrix entry types ---
    R = []
    for row in rref_matrix:
        new_row = []
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, Fraction)):
                raise TypeError(
                    "rref_matrix entries must be int or Fraction (no float/bool)"
                )
            new_row.append(Fraction(x))
        R.append(new_row)

    # --- tier 3: pivot_cols well-formedness as a plain list of indices ---
    if not isinstance(pivot_cols, (list, tuple)):
        raise TypeError("pivot_cols must be a list (or tuple) of int")
    for c in pivot_cols:
        if isinstance(c, bool) or not isinstance(c, int):
            raise TypeError("pivot_cols entries must be Python int (bool excluded)")
    pivot_cols = list(pivot_cols)

    if len(set(pivot_cols)) != len(pivot_cols):
        raise ValueError("pivot_cols must not contain a duplicate column index")
    if pivot_cols != sorted(pivot_cols):
        raise ValueError("pivot_cols must be sorted in strictly increasing order")
    for c in pivot_cols:
        if not (0 <= c < n):
            raise ValueError(f"pivot column {c} out of range for a {m}x{n} matrix")
    if len(pivot_cols) > m:
        raise ValueError("pivot_cols cannot have more entries than rref_matrix has rows")

    # --- tier 4: rref_matrix must genuinely, structurally, match pivot_cols
    #     (enumerated conditions 1-4 in the Subproblem 2 prompt) ---
    for i, p in enumerate(pivot_cols):
        if R[i][p] != 1:
            raise ValueError(
                f"rref_matrix[{i}][{p}] = {R[i][p]}, expected exactly 1 "
                f"(row {i} is supposed to be the pivot row for column {p})"
            )
        for j in range(p):
            if R[i][j] != 0:
                raise ValueError(
                    f"rref_matrix[{i}][{j}] = {R[i][j]} is nonzero and lies left of "
                    f"row {i}'s declared pivot column {p}; the pivot must be the "
                    f"leftmost nonzero entry of its row (condition 2)"
                )
        for j in range(m):
            if j != i and R[j][p] != 0:
                raise ValueError(
                    f"pivot column {p} has a nonzero entry {R[j][p]} in row {j}, "
                    f"which is not column {p}'s pivot row ({i})"
                )
    for j in range(len(pivot_cols), m):
        if any(x != 0 for x in R[j]):
            raise ValueError(
                f"row {j} is beyond the declared rank {len(pivot_cols)} "
                f"(len(pivot_cols)) but is not all zero"
            )

    # --- the free-variable construction itself ---
    pivot_set = set(pivot_cols)
    free_cols = [c for c in range(n) if c not in pivot_set]

    basis = []
    for free in free_cols:
        v = [Fraction(0)] * n
        v[free] = Fraction(1)
        for i, p in enumerate(pivot_cols):
            v[p] = -R[i][free]
        basis.append(v)

    return basis

# Main Problem

## Prompt

Implement `general_solution(matrix, b)` — the complete solution set of the linear system $Ax = b$, given `matrix` (representing $A$) and the right-hand side `b`, expressed exactly as a **particular solution plus a null-space basis**.

By the Rouché–Capelli theorem, $Ax = b$ is consistent exactly when $\operatorname{rank}(A) = \operatorname{rank}([A \mid b])$, where $[A \mid b]$ is $A$ with $b$ appended as one extra column. Equivalently — and this is the form to implement — row-reduce the augmented matrix $[A \mid b]$: the system is **inconsistent** exactly when the augmented column (column index $n$, where $n$ is the number of columns of $A$) turns out to be a pivot column of $\operatorname{rref}([A \mid b])$, because that means some row reduced to $0 = 1$ (a row of all zeros in the $A$-part, and a nonzero entry in the augmented column).

When the system **is** consistent, the general solution is $x = x_p + \sum_i t_i\, v_i$ for arbitrary rationals $t_i$, where $x_p$ is any one particular solution and $\{v_i\}$ is a basis of $\ker A$. Read $x_p$ directly off $\operatorname{rref}([A\mid b])$: set every free variable of $A$ to $0$, and for each pivot column $p_i$ of $A$ (row $i$ of the reduced augmented matrix), $x_{p,\,p_i}$ is exactly the augmented-column entry in that row. The pivot columns of $\operatorname{rref}([A \mid b])$ restricted to the first $n$ columns are exactly the pivot columns of $\operatorname{rref}(A)$ — row-reducing the augmented matrix performs the same row operations on the $A$-block as row-reducing $A$ alone would — so a single call to `rref` on the augmented matrix supplies everything both helpers below need; **there is no need to call `rref` a second time on `matrix` by itself.**

**Helper contracts.** Both helpers below are already implemented, tested, and correctly integrated into the reference solution. Their exact contracts are restated here so this prompt is self-contained.

```python
def rref(matrix):
    """
    Returns (rref_matrix, pivot_cols).

    rref_matrix : list of list of Fraction
        The reduced row echelon form of matrix (leading 1 in each
        nonzero row's leftmost entry, that column cleared everywhere
        else, all-zero rows at the bottom). Same shape as the input.
    pivot_cols : list of int
        The pivot columns, strictly increasing. len(pivot_cols) is the
        rank of matrix.
    Raises ValueError if matrix is empty, if any row is empty, or if
    rows have differing lengths.
    Raises TypeError if any entry is not int or Fraction (includes
    bool and float).
    """


def null_space_basis(rref_matrix, pivot_cols):
    """
    Returns basis: list of list of Fraction.

    One vector per free column of rref_matrix, in increasing order of
    the free column it corresponds to, each of length n (the number of
    columns). Empty list when rref_matrix has full column rank.
    Raises TypeError if an entry of rref_matrix is not int or Fraction,
    or if a pivot_cols entry is not a Python int (bool excluded).
    Raises ValueError if rref_matrix is empty/ragged, or if pivot_cols
    contains a duplicate, an out-of-range index, is unsorted, or has
    more entries than rref_matrix has rows.
    """


def general_solution(matrix, b):
    """
    Parameters
    ----------
    matrix : list of list
        Non-empty m x n matrix A, m >= 1, n >= 1, every row length n.
        Entries must be int or Fraction.
    b : list
        Length-m right-hand side. Entries must be int or Fraction.

    Returns
    -------
    None
        If Ax = b is inconsistent (no solution exists).
    (particular, basis) : tuple
        Otherwise: particular is a length-n list of Fraction giving
        one exact solution; basis is a list of length-n lists of
        Fraction, a basis of ker(A) (possibly empty, if A has full
        column rank). The full solution set is
        { [particular[i] + sum(t_k * basis[k][i] for k in range(len(basis)))
           for i in range(n)] : t_k rational }.

    Raises
    ------
    ValueError
        If matrix is empty, if any row of matrix is empty, if rows of
        matrix have differing lengths, or if len(b) != number of rows
        of matrix.
    TypeError
        If any entry of matrix or b is not int or Fraction (this
        includes bool and float), or if b is not a list/tuple.

    Precedence, when more than one problem is present at once: matrix's
    own shape (non-empty, rectangular) is fully checked first. Only
    then is b checked -- its type, then its length against matrix,
    then its entries' types, in that order. matrix's *entries* are not
    separately re-checked in this function; they surface (as TypeError
    or ValueError from rref) only once the augmented matrix is built,
    which happens after every b check above has already passed. So a
    call with both a bad matrix entry and a bad b entry raises the
    error about b, not the one about matrix.
    """
    ...
    return None  # or (particular, basis)
```

## Background

**The three outcomes.** Every linear system $Ax=b$ falls into exactly one of three cases, and `general_solution` must distinguish all three correctly. **Consistency is decided first**: the system has a solution if and only if $b \in \operatorname{col}(A)$, equivalently $\operatorname{rank}([A\mid b]) = \operatorname{rank}(A)$ (Rouché–Capelli). The rank of $A$ alone governs the *number* of solutions only *once the system is known to be consistent* — full column rank or rank deficiency says nothing on its own about solvability.

- **No solution (inconsistent).** $b$ is not in the column space of $A$ ($\operatorname{rank}([A\mid b]) = \operatorname{rank}(A)+1$); `general_solution` returns `None`. This can happen for **any** shape and **any** rank of $A$, including full column rank.
- **Exactly one solution.** The system is consistent **and** $A$ has full column rank ($\operatorname{rank}(A) = n$); then `ker A` is trivial, so `basis == []` and `particular` is the unique solution. (Full column rank alone is not enough — an inconsistent full-column-rank system, e.g. $A=\begin{pmatrix}1\\1\end{pmatrix}$, $b=(1,2)$, has no solution.)
- **Infinitely many solutions.** The system is consistent **and** $A$ has rank less than $n$; then the solution set is the affine subspace $x_p + \ker A$, a "shifted" copy of the null space passing through $x_p$, of dimension $n - \operatorname{rank}(A)$. (Rank $< n$ alone is not enough — an inconsistent rank-deficient system, e.g. $x+y+z=3$ and $2x+2y+2z=7$, has no solution despite $\operatorname{rank}(A)=1 < 3$.)

**Why checking the augmented pivot column is enough.** $\operatorname{rank}([A\mid b])$ can only ever be $\operatorname{rank}(A)$ or $\operatorname{rank}(A)+1$ — appending one column raises the rank by at most one. It is exactly $\operatorname{rank}(A)+1$ when the new column is itself a pivot column of the reduced augmented matrix, and by Rouché–Capelli that is precisely the inconsistent case. So testing "is column $n$ a pivot column?" is a single, cheap, exact check — no separate rank comparison, and no second call to `rref`, is needed.

**Scientific trap: row/column counts do not determine which of the three outcomes occurs — only rank does.** Two matching misconceptions are both false, in opposite directions:

- *"More equations than unknowns ($m > n$) means no solution, or at best a unique one."* False: if some of those extra equations are linear combinations of the others (rank $< m$), the system can still have infinitely many solutions. Three equations in two unknowns, where the third is the sum of the first two, is consistent and one-dimensional-family-free (in fact unique here) — or, with one more dependent row than that, can have a whole line of solutions — regardless of $m=3 > n=2$.
- *"Fewer equations than unknowns ($m < n$) means infinitely many solutions."* Also false: two equations in three unknowns are individually "underdetermined," but if the second equation is a multiple of the first *coefficient-wise* while disagreeing on the right-hand side (e.g. $x+y+z=3$ and $2x+2y+2z=7$, where the second should equal $6$, not $7$, if it's twice the first), the system is flatly inconsistent — $m=2 < n=3$, yet there is no solution at all, not even one.

The only reliable test is the one this contract already implements: is the augmented column a pivot column of $\operatorname{rref}([A\mid b])$? Rank, not row or column count, is what governs consistency and solution-set dimension — this is worth internalizing because "more equations, more constrained" is an intuition that works for *generic* (randomly chosen) systems but fails as soon as any row happens to be a linear combination of the others, which is exactly the case that makes testing real mathematical understanding (rather than pattern-matching on matrix shape) possible.

**Worked example (infinitely many solutions).** For
$$
A = \begin{pmatrix}1 & 1 & 1\end{pmatrix}, \qquad b = (3),
$$
the augmented matrix is already in RREF: $\begin{pmatrix}1 & 1 & 1 & \big| & 3\end{pmatrix}$, pivot column $\{0\}$. Column $3$ (the augmented column) is not a pivot column, so the system is consistent. The particular solution sets the free columns ($1, 2$) to $0$: $x_p = (3, 0, 0)$. The null space of $A$ has two basis vectors (one per free column), $(-1,1,0)$ and $(-1,0,1)$, so the general solution is $(3,0,0) + t_1(-1,1,0) + t_2(-1,0,1)$ for any rationals $t_1, t_2$ — a plane's worth of solutions, as expected from one equation in three unknowns.

**A genuinely independent check.** Once `general_solution` returns `(particular, basis)`, nothing about how they were derived is assumed further: substituting back, $A \cdot \text{particular}$ must equal $b$ exactly, and $A \cdot v$ must equal the zero vector exactly for every $v$ in `basis`. This never touches `rref` or `null_space_basis` internals — it is pure matrix-vector multiplication against the *original* `matrix` and `b` — so it catches a mistake in either helper, or in how the Main Problem combines them, that self-referential testing could miss.

**Reusing one `rref` call.** Row-reducing $[A\mid b]$ performs exactly the same sequence of row operations on the first $n$ columns as row-reducing $A$ alone would (the extra column along for the ride doesn't influence which row operations are chosen, since pivot search and elimination only ever look at the columns being processed). So the first $n$ columns of $\operatorname{rref}([A\mid b])$ *are* $\operatorname{rref}(A)$, with the same pivot columns — one call to `rref` on the augmented matrix is both necessary and sufficient.

**Common bugs.**

- Calling `rref` twice — once on `matrix` alone and once on the augmented matrix — instead of reusing the single augmented call. Not wrong, just wasted work, and a sign the "why one call suffices" reasoning above wasn't used.
- Checking `rank(A) == rank([A|b])` by comparing two separately-computed rank values instead of the simpler, equivalent "is column $n$ a pivot column" test — easy to get an off-by-one on which matrix's pivot list is which.
- Forgetting to slice the augmented column back off before calling `null_space_basis` (passing an $m \times (n+1)$ matrix through unchanged corrupts the free-column count by one).
- Setting free variables in the particular solution to something other than exactly `Fraction(0)`.
- Returning `(None, None)` instead of plain `None` for an inconsistent system, or vice versa, silently breaking every caller's `is None` check.

## Testing Template

In [ ]:
# ============ TEST CASES ============

from fractions import Fraction


def _matvec(A, v):
    return [sum(Fraction(a) * Fraction(x) for a, x in zip(row, v)) for row in A]


def _check_solution(matrix, b, result):
    """Independent check: A @ particular == b and A @ v == 0 for every basis vector."""
    assert result is not None, "expected a solution, got None"
    particular, basis = result
    got_b = _matvec(matrix, particular)
    assert got_b == [Fraction(x) for x in b], (
        f"A @ particular = {got_b}, expected {b}"
    )
    for v in basis:
        w = _matvec(matrix, v)
        assert all(x == 0 for x in w), f"A @ {v} = {w}, expected all zeros"


def test_case_1():
    A, b = [[1, 2], [3, 4]], [5, 6]
    result = general_solution(A, b)
    assert result == ([Fraction(-4), Fraction(9, 2)], []), f"got {result}"
    _check_solution(A, b, result)
    print('test_case_1 passed')


def test_case_2():
    A, b = [[1, 1, 1]], [3]
    result = general_solution(A, b)
    particular, basis = result
    assert particular == [3, 0, 0], f"got {particular}"
    assert basis == [[-1, 1, 0], [-1, 0, 1]], f"got {basis}"
    _check_solution(A, b, result)
    print('test_case_2 passed')


def test_case_3():
    A, b = [[1, 1], [2, 2]], [1, 3]
    assert general_solution(A, b) is None
    print('test_case_3 passed')


def test_case_4():
    A, b = [[1, 1], [2, 2], [1, -1]], [4, 8, 0]
    result = general_solution(A, b)
    assert result == ([Fraction(2), Fraction(2)], []), f"got {result}"
    _check_solution(A, b, result)
    print('test_case_4 passed')


def test_case_5():
    A, b = [[2, 1, 1], [1, 3, 2]], [4, 5]
    result = general_solution(A, b)
    particular, basis = result
    assert particular == [Fraction(7, 5), Fraction(6, 5), 0], f"got {particular}"
    assert basis == [[Fraction(-1, 5), Fraction(-3, 5), 1]], f"got {basis}"
    _check_solution(A, b, result)
    print('test_case_5 passed')


def test_case_6():
    A, b = [[1, 2, 3], [2, 4, 6]], [0, 0]
    result = general_solution(A, b)
    particular, basis = result
    assert particular == [0, 0, 0], f"got {particular}"
    assert len(basis) == 2, f"got {len(basis)}"
    _check_solution(A, b, result)
    print('test_case_6 passed')


def test_case_7():
    A, b = [[5]], [10]
    result = general_solution(A, b)
    assert result == ([Fraction(2)], []), f"got {result}"
    _check_solution(A, b, result)
    print('test_case_7 passed')


def test_case_8():
    # rank-nullity: len(basis) == n - rank(A), rank taken independently via rref
    cases = [
        ([[1, 2], [3, 4]], [5, 6]),
        ([[1, 1, 1]], [3]),
        ([[1, 1], [2, 2], [1, -1]], [4, 8, 0]),
        ([[2, 1, 1], [1, 3, 2]], [4, 5]),
        ([[1, 2, 3], [2, 4, 6]], [0, 0]),
        ([[5]], [10]),
    ]
    for A, b in cases:
        n = len(A[0])
        _, rank_pc = rref(A)
        _, basis = general_solution(A, b)
        assert len(basis) == n - len(rank_pc), (
            f"{A}: len(basis)={len(basis)}, expected {n - len(rank_pc)}"
        )
    print('test_case_8 passed')


def test_case_9():
    # m < n does not imply a solution exists: two parallel equations disagree
    A, b = [[1, 1, 1], [2, 2, 2]], [3, 7]
    assert general_solution(A, b) is None, (
        "m < n does not guarantee consistency; only rank does"
    )
    print('test_case_9 passed')


def test_case_10():
    # m > n does not imply no solution: redundant row agrees -> consistent;
    # changing only the redundant RHS -> inconsistent
    A = [[1, 2], [3, 4], [4, 6]]
    result = general_solution(A, [5, 6, 11])
    assert result is not None, "m > n does not guarantee no solution"
    _check_solution(A, [5, 6, 11], result)
    assert general_solution(A, [5, 6, 12]) is None, (
        "changing only the redundant row's RHS must flip consistency"
    )
    print('test_case_10 passed')


def test_case_11():
    # shape validation -> ValueError
    for bad in [[], [[]], [[1, 2], [1]]]:
        try:
            general_solution(bad, [1])
            assert False, f"expected ValueError for {bad}"
        except ValueError:
            pass
    try:
        general_solution([[1, 2], [3, 4]], [1, 2, 3])
        assert False, "expected ValueError for mismatched b length"
    except ValueError:
        pass
    print('test_case_11 passed')


def test_case_12():
    # type validation -> TypeError
    for bad_M in [[[1.0, 2], [3, 4]], [[True, 2], [3, 4]]]:
        try:
            general_solution(bad_M, [1, 2])
            assert False, f"expected TypeError for matrix={bad_M}"
        except TypeError:
            pass
    for bad_b in [[1.0, 2], [True, 2]]:
        try:
            general_solution([[1, 2], [3, 4]], bad_b)
            assert False, f"expected TypeError for b={bad_b}"
        except TypeError:
            pass
    print('test_case_12 passed')


def test_case_13():
    # helper contracts used directly, exactly as the prompt restates them
    R, pc = rref([[1, 1, 1]])
    assert R == [[1, 1, 1]], f"got {R}"
    assert pc == [0], f"got {pc}"
    basis = null_space_basis(R, pc)
    assert basis == [[-1, 1, 0], [-1, 0, 1]], f"got {basis}"
    print('test_case_13 passed')


def test_case_14():
    # Precedence: b's own type/length are checked before matrix's entries.
    # A float in matrix with a wrong-length b must raise ValueError
    # (about b's length), not TypeError (about matrix's entries).
    try:
        general_solution([[1.0, 2], [3, 4]], [1])
        assert False, "expected ValueError"
    except ValueError:
        pass
    except TypeError:
        assert False, (
            "TypeError fired for matrix's float entry; b's length must be "
            "checked before matrix's entries"
        )
    print('test_case_14 passed')


def test_case_15():
    # Precedence: matrix's own shape (empty / ragged) is checked before b's
    # type. A ragged matrix with a scalar b must raise ValueError (about
    # matrix's shape), not TypeError (about b's type).
    try:
        general_solution([[1, 2], [1]], 5)
        assert False, "expected ValueError"
    except ValueError:
        pass
    except TypeError:
        assert False, (
            "TypeError fired for b; matrix's ragged shape must be "
            "checked before b's type"
        )
    print('test_case_15 passed')

## Solution

In [ ]:
from fractions import Fraction


def rref(matrix):
    """Inlined from Subproblem 1: returns (rref_matrix, pivot_cols)."""
    if not isinstance(matrix, (list, tuple)) or len(matrix) == 0:
        raise ValueError("matrix must be a non-empty list of rows")

    n_cols = None
    for row in matrix:
        if not isinstance(row, (list, tuple)) or len(row) == 0:
            raise ValueError("each row must be a non-empty list")
        if n_cols is None:
            n_cols = len(row)
        elif len(row) != n_cols:
            raise ValueError("all rows must have the same length")

    m, n = len(matrix), n_cols

    R = []
    for row in matrix:
        new_row = []
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, Fraction)):
                raise TypeError(
                    "matrix entries must be int or Fraction (no float/bool)"
                )
            new_row.append(Fraction(x))
        R.append(new_row)

    pivot_cols = []
    r = 0
    for c in range(n):
        pivot = None
        for i in range(r, m):
            if R[i][c] != 0:
                pivot = i
                break
        if pivot is None:
            continue

        R[r], R[pivot] = R[pivot], R[r]
        pv = R[r][c]
        R[r] = [x / pv for x in R[r]]

        for i in range(m):
            if i != r and R[i][c] != 0:
                factor = R[i][c]
                R[i] = [a - factor * b for a, b in zip(R[i], R[r])]

        pivot_cols.append(c)
        r += 1
        if r == m:
            break

    return R, pivot_cols


def null_space_basis(rref_matrix, pivot_cols):
    """Inlined from Subproblem 2 (identical contract, including the four-tier
    validation): returns basis: list of list of Fraction."""
    if not isinstance(rref_matrix, (list, tuple)) or len(rref_matrix) == 0:
        raise ValueError("rref_matrix must be a non-empty list of rows")

    n_cols = None
    for row in rref_matrix:
        if not isinstance(row, (list, tuple)) or len(row) == 0:
            raise ValueError("each row of rref_matrix must be a non-empty list")
        if n_cols is None:
            n_cols = len(row)
        elif len(row) != n_cols:
            raise ValueError("all rows of rref_matrix must have the same length")

    m, n = len(rref_matrix), n_cols

    R = []
    for row in rref_matrix:
        new_row = []
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, Fraction)):
                raise TypeError(
                    "rref_matrix entries must be int or Fraction (no float/bool)"
                )
            new_row.append(Fraction(x))
        R.append(new_row)

    if not isinstance(pivot_cols, (list, tuple)):
        raise TypeError("pivot_cols must be a list (or tuple) of int")
    for c in pivot_cols:
        if isinstance(c, bool) or not isinstance(c, int):
            raise TypeError("pivot_cols entries must be Python int (bool excluded)")
    pivot_cols = list(pivot_cols)

    if len(set(pivot_cols)) != len(pivot_cols):
        raise ValueError("pivot_cols must not contain a duplicate column index")
    if pivot_cols != sorted(pivot_cols):
        raise ValueError("pivot_cols must be sorted in strictly increasing order")
    for c in pivot_cols:
        if not (0 <= c < n):
            raise ValueError(f"pivot column {c} out of range for a {m}x{n} matrix")
    if len(pivot_cols) > m:
        raise ValueError("pivot_cols cannot have more entries than rref_matrix has rows")

    # --- tier 4: rref_matrix must genuinely, structurally, match pivot_cols
    #     (enumerated conditions 1-4 in the Subproblem 2 prompt) ---
    for i, p in enumerate(pivot_cols):
        if R[i][p] != 1:
            raise ValueError(
                f"rref_matrix[{i}][{p}] = {R[i][p]}, expected exactly 1 "
                f"(row {i} is supposed to be the pivot row for column {p})"
            )
        for j in range(p):
            if R[i][j] != 0:
                raise ValueError(
                    f"rref_matrix[{i}][{j}] = {R[i][j]} is nonzero and lies left of "
                    f"row {i}'s declared pivot column {p}; the pivot must be the "
                    f"leftmost nonzero entry of its row (condition 2)"
                )
        for j in range(m):
            if j != i and R[j][p] != 0:
                raise ValueError(
                    f"pivot column {p} has a nonzero entry {R[j][p]} in row {j}, "
                    f"which is not column {p}'s pivot row ({i})"
                )
    for j in range(len(pivot_cols), m):
        if any(x != 0 for x in R[j]):
            raise ValueError(
                f"row {j} is beyond the declared rank {len(pivot_cols)} "
                f"(len(pivot_cols)) but is not all zero"
            )

    pivot_set = set(pivot_cols)
    free_cols = [c for c in range(n) if c not in pivot_set]

    basis = []
    for free in free_cols:
        v = [Fraction(0)] * n
        v[free] = Fraction(1)
        for i, p in enumerate(pivot_cols):
            v[p] = -R[i][free]
        basis.append(v)

    return basis


def general_solution(matrix, b):
    if not isinstance(matrix, (list, tuple)) or len(matrix) == 0:
        raise ValueError("matrix must be a non-empty list of rows")

    n = None
    for row in matrix:
        if not isinstance(row, (list, tuple)) or len(row) == 0:
            raise ValueError("each row of matrix must be a non-empty list")
        if n is None:
            n = len(row)
        elif len(row) != n:
            raise ValueError("all rows of matrix must have the same length")

    if not isinstance(b, (list, tuple)):
        raise TypeError("b must be a list (or tuple) of int or Fraction")
    if len(b) != len(matrix):
        raise ValueError("b must have exactly one entry per row of matrix")
    for x in b:
        if isinstance(x, bool) or not isinstance(x, (int, Fraction)):
            raise TypeError("b entries must be int or Fraction (no float/bool)")

    # Row-reduce [A | b] once; its first n columns are rref(A) with the
    # same pivot columns, so a second call to rref(matrix) is unnecessary.
    augmented = [list(row) + [bi] for row, bi in zip(matrix, b)]
    R, pcols = rref(augmented)

    if n in pcols:
        return None  # a row reduced to 0 = (nonzero): inconsistent

    particular = [Fraction(0)] * n
    for i, p in enumerate(pcols):
        particular[p] = R[i][n]

    basis = null_space_basis([row[:n] for row in R], pcols)

    return particular, basis